# Chapter 1: Chunking

**Estimated time: 15 minutes**

Your RAG system returns the wrong answer and you blame the model. But the real problem happened before any search ran. You chunked your documents wrong.

In this notebook you will split the same refund policy two different ways and watch the same question produce two very different answers from the LLM. The only thing that changes is one number. Chunk size.

Run every cell top to bottom. When you reach the "Try it yourself" section, change the numbers and rerun.

## What you will see in three minutes

You are going to ask this question:

> How much of my subscription fee is refunded if I cancel my Pro Annual plan after five months of use?

You will run it through two versions of the same pipeline.

- **Version A** chunks each document into 1500 character blocks. This is the default from most RAG quickstart tutorials. The retrieval score is mediocre (around 0.8 under L2 distance) and the LLM gives a verbose six-sentence breakdown to get to the answer.
- **Version B** chunks the same documents into 500 character blocks. The retrieval score drops to around 0.6, a twenty-five percent improvement, and the LLM gives a single clean sentence with the exact numbers.

Same documents. Same question. Same embedding model. Same LLM. One number changes. The retrieval improves measurably and the answer style flips from textbook to production ready.

That is the chapter. Now let us get set up.

## One time setup, getting your API keys

This is the only chapter that walks through API keys. Every other chapter assumes the keys are already in Colab secrets and picks them up automatically.

You need three keys. All three have free tiers that cover the entire series. Total cost across all seven chapters is under two dollars.

1. **OpenAI** at [platform.openai.com](https://platform.openai.com). New accounts get five dollars of free credit. Copy your key.
2. **LangSmith** at [smith.langchain.com](https://smith.langchain.com). Sign up, go to Settings, API Keys, create a personal key. Free tier gives you five thousand traces per month.
3. **Cohere** at [dashboard.cohere.com](https://dashboard.cohere.com). Sign up, go to API Keys, copy your trial key. Free tier gives you one thousand rerank calls per month. You will not need this until Chapter 5, but add it now so every notebook just works.

Now add them to Colab secrets. Click the **key icon** on the left sidebar of Colab. Click **+ Add new secret** three times and add:

- `OPENAI_API_KEY` set to your OpenAI key
- `LANGCHAIN_API_KEY` set to your LangSmith key
- `COHERE_API_KEY` set to your Cohere key

For each one, toggle **Notebook access** on. You only do this once. Every chapter from now on picks the keys up automatically.

## Install the dependencies

Run the next cell once. It installs the Python packages the notebook needs.

If you are running this in Google Colab, the cell detects Colab and pip installs everything fresh. If you are running locally in Jupyter, the cell skips the install (it assumes you already ran `pip install -r requirements.txt` in your virtual environment). Either way, you just click Run and keep going.

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Colab detected. Installing dependencies (takes about 60 seconds).")
    !pip install -q \
        langchain==0.3.14 \
        langchain-openai==0.2.14 \
        langchain-community==0.3.14 \
        langchain-text-splitters==0.3.5 \
        faiss-cpu==1.13.2 \
        pypdf==5.1.0 \
        langsmith==0.2.6 \
        pandas==2.2.3
    print("Done.")
else:
    print("Local environment detected. Skipping pip install.")
    print("Make sure you have run 'pip install -r requirements.txt' in your venv.")

In [ ]:
import os, sys

if IN_COLAB:
    # Colab: clone the repo so the notebook can import from utils and load the corpus.
    if not os.path.exists("rag-for-pms"):
        !git clone -q https://github.com/DDRXV/rag-for-pms.git
    os.chdir("rag-for-pms")
else:
    # Local: you are already inside the repo. Walk up to the repo root if needed.
    if os.path.basename(os.getcwd()) == "chapters":
        os.chdir("..")

sys.path.insert(0, os.getcwd())
print(f"Working directory: {os.getcwd()}")

In [ ]:
from utils.shared import get_keys
get_keys()

## What you just did

`get_keys` pulled your three API keys out of Colab secrets and turned on LangSmith tracing. From now on every embedding call, every chat call, and every retrieval step gets logged automatically to a clickable waterfall trace at [smith.langchain.com](https://smith.langchain.com).

## Look at the corpus

You are about to load six SkillAgents AI documents. The star of this chapter is `refund_policy.pdf`. The other five are there to make the retrieval problem realistic.

In [ ]:
from utils.shared import load_corpus

docs = load_corpus()

for d in docs[:6]:
    print(f"  {d.metadata.get('source', 'unknown'):25s}  {len(d.page_content):5d} chars")

## Peek at Section 4.2

Before we run anything, let us look at the exact text that answers the question. This lives inside `refund_policy.pdf`.

In [ ]:
refund = next(d for d in docs if d.metadata["source"] == "refund_policy.pdf")

start = refund.page_content.find("4.2")
print(refund.page_content[start:start + 650])

This is the target. When we run our question through the pipeline, we want the top retrieved chunk to be this exact passage. Anything else is a miss.

Notice how short it is. Two paragraphs. Roughly five hundred characters. Remember that number, it comes back in a minute.

## Version A: the default from the quickstart

Most RAG quickstart tutorials suggest a chunk size around 1000 to 2000 characters. Let us pick 1500, which sounds reasonable. Each document gets split into blocks of roughly 1500 characters. The refund policy has 2617 characters total, so it becomes two chunks.

Then we search with our question and see what comes back.

In [ ]:
from utils.shared import build_index, search, show_results

# A common default from RAG tutorials online. Sounds reasonable. Let us see.
chunk_size = 1500

default_index = build_index(chunk_size=chunk_size, chunk_overlap=0)
question = "How much of my subscription fee is refunded if I cancel my Pro Annual plan after five months of use?"

default_results = search(default_index, question, k=3)
show_results(default_results, question=question)

## What went wrong

The top row points at `refund_policy.pdf`. Good, the retriever found the right document. But look at the score column.

**Reading L2 distance scores.** Under L2 distance, lower is better. In this notebook the useful range runs from about 0.5 for a direct semantic match to about 1.2 for an unrelated chunk. Anything above 1.0 is effectively off-topic. Our top score here is around 0.8. Not terrible. Not good.

Why is the score mediocre? Because the 1500 character chunk mashes Section 4.1 (cancellation process), Section 4.2 (the actual pro-rated formula), Section 5 (Team plan rules), and part of Section 6 (Enterprise) into one undifferentiated block. The embedding for that block is an average of all those topics. The question about Pro Annual refunds pulls on the average, not on the specific Section 4.2 content.

An LLM given this chunk has to find Section 4.2 inside 1500 characters of mixed policy text and then do the math. It might get it right. It might hedge. It might hallucinate the wrong fee.

## Version B: paragraph-sized chunks

Now rebuild the index with `chunk_size=500` and a small overlap. The same refund policy becomes several smaller pieces, each roughly the size of one paragraph. Remember the Section 4.2 passage was about 500 characters long. That is not an accident.

When we search the same question, the match should land directly on the pro-rated calculation.

In [ ]:
# Paragraph-sized chunks. Each one holds one focused idea, not five.
chunk_size = 500
chunk_overlap = 50

focused_index = build_index(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
focused_results = search(focused_index, question, k=3)
show_results(focused_results, question=question)

## What got better

Look at the top score. It should drop from around 0.8 to around 0.6. That is a real retrieval improvement. The preview column shows the exact pro-rated calculation text from Section 4.2 with almost nothing else in the way.

Same document. Same question. Same embedding model. Different chunk size. Cleaner signal.

Now let us see if this actually changes the LLM's answer.

## The moment that matters, what the LLM actually says

So far we have been looking at distance scores. A PM does not care about distance scores. A PM cares about the answer the customer sees. Let us feed both sets of chunks to `gpt-4o-mini` and compare.

In [ ]:
from utils.shared import generate_answer

answer_default = generate_answer(default_results, question)
answer_focused = generate_answer(focused_results, question)

print("VERSION A (chunk_size=1500)")
print("-" * 60)
print(answer_default)
print()
print("VERSION B (chunk_size=500)")
print("-" * 60)
print(answer_focused)

## Read both answers carefully

Both answers will land on $824.42. Modern language models like `gpt-4o-mini` are strong enough to reason their way to the right number from mildly noisy context. So the lesson is not "one answer is wrong and the other is right". The lesson is more interesting than that.

Look at **how each answer is written**.

Version A is probably a step by step breakdown. The LLM walks you through the consumed months, the pro-rated fraction, the multiplication, the processing fee, and the final amount, one sentence per step. Six sentences. That is what happens when the LLM receives 1500 characters of mixed policy text and has to pick out the relevant parts. It shows its work because it had to do the work.

Version B is probably one clean sentence. "$824.42, which is a pro-rated refund of $874.42 minus a $50 processing fee." Direct. Production ready. No preamble. That is what happens when the LLM receives a focused 500 character chunk that already isolates the answer. It does not need to narrate the reasoning because the chunk did the reasoning for it.

This is the actual product difference. In a customer facing chatbot you ship Version B, not Version A. Customers do not want a step by step recap. They want the number and the formula. Cleaner retrieval produces cleaner answers without any prompt engineering. The retrieval layer is the single biggest lever on answer quality, and chunk size is the single biggest lever on the retrieval layer.

## Open your LangSmith trace

If you want to see every step inside the pipeline, open [smith.langchain.com](https://smith.langchain.com), click into the project called `rag-for-pms`, and open the most recent trace. You will see two retrieval calls and two generation calls, side by side, with every chunk, every score, every token count, and every millisecond of latency.

Optional but worth five minutes.

## What you can do at work on Monday

You do not need to remember chunk size numbers. You need to remember the question.

When you review a RAG feature your team is building, ask three things.

1. What chunk size did you pick and why?
2. Show me the top three retrieved chunks for a real user question.
3. What happens if the top chunk is the wrong section of the right document?

If the team cannot answer these cleanly, the product will fail in production. Most RAG failures live in the retrieval layer, not the LLM, and chunk size is the single biggest lever in the retrieval layer.

## Try it yourself

Pick any of these. Change the number in the cell above, rerun, watch what happens.

1. Change `chunk_size` in the Version B cell to **300**. Rerun the search and the LLM answer cells. See if the score and the answer get better, stay the same, or get worse.
2. Change `chunk_size` to **5000**. Rerun everything. Watch the score climb back toward 1.0 and the LLM answer lose precision.
3. Change the `question` variable to `"What is the price of the Team plan?"`. Rerun the Version B cells. Watch the top match switch from `refund_policy.pdf` to `pricing.pdf` without changing any other code.

## Homework

Two exercises you can do on your own. Each one takes about fifteen minutes.

1. **Try `MarkdownHeaderTextSplitter`.** Look it up in the LangChain docs. Use it on `data/skillagents/product_guide.md` to split by H2 headers instead of character count. Print the first three chunks it produces. Do the chunks look cleaner or messier than the default splitter?

2. **Apply it to your own company.** Pick one real document from your own product. A refund policy, a pricing page, a feature guide. Drop it into `data/skillagents/` alongside the others. Rerun the notebook with a real question from a real user. Does the retrieval work? If not, what chunk size would you try next? Share your before-and-after in the cohort Slack.

## What is next

In Chapter 2 your chunking is perfect, your index is clean, and your vector search still returns the wrong document. Why? Because the user asked a vague question and your pipeline searched for the exact wrong thing.

You are going to fix this by rewriting the question before it ever touches the vector store. The technique is called query translation, and it is the single highest leverage move you can make on a production RAG system.

See you in Chapter 2.